<a href="https://colab.research.google.com/github/roy2393/ragstack/blob/rr%2Fmain/notebooks/rr_capstone_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone -b rr/main https://github.com/roy2393/ragstack.git
%cd ragstack

Cloning into 'ragstack'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 43 (delta 5), reused 39 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 1.04 MiB | 7.47 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/ragstack


In [ ]:
!git status

On branch rr/main
Your branch is up to date with 'origin/rr/main'.

nothing to commit, working tree clean


In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

# Load specifically the CUAD (Legal) dataset from RAGBench
cuad_dataset = load_dataset("rungalileo/ragbench", "cuad")

# Print the structure to see the train, validation, and test splits
print(cuad_dataset)

# View the first example in the training set to understand the schema
print(cuad_dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

cuad/train-00000-of-00001.parquet:   0%|          | 0.00/56.4M [00:00<?, ?B/s]

cuad/validation-00000-of-00001.parquet:   0%|          | 0.00/15.7M [00:00<?, ?B/s]

cuad/test-00000-of-00001.parquet:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/510 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/510 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score'],
        num_rows: 1530
    })
    validation: Dataset({
        features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'rel

In [ ]:
# Take just the first 100 examples from the training set
small_cuad_sample = cuad_dataset['train'].select(range(100))

print(f"Working with {len(small_cuad_sample)} documents for prototyping.")

print(small_cuad_sample[0])

Working with 100 documents for prototyping.
{'id': 'MOELIS_CO_03_24_2014-EX-10.19-STRATEGIC ALLIANCE AGREEMENT__Source Code Escrow', 'question': 'Is one party required to deposit its source code into escrow with a third party, which can be released to the counterparty upon the occurrence of certain events (bankruptcy,\xa0 insolvency, etc.)?', 'documents': ['Exhibit 10.19   STRATEGIC ALLIANCE AGREEMENT\n\nAmong   SUMITOMO MITSUI BANKING CORPORATION, SMBC NIKKO SECURITIES INC.   And   MOELIS & COMPANY HOLDINGS LP, MOELIS & COMPANY HOLDINGS GP LLC   Dated December 27, 2011\n\n\n\n\n\n  TABLE OF CONTENTS                                               ARTICLE I.       CERTAIN DEFINITIONS; INTERPRETATION.       1.1   Certain Definitions   2   1.2   Interpretations   5               ARTICLE II.       STRATEGIC ALLIANCE.       2.1   Strategic Alliance   6   2.2   Obligations of the Parties   6               ARTICLE III.       SCOPE.       3.1   Scope   6   3.2   Covered Businesses   6   3.3   C

Start Chunking

In [ ]:
!pip install langchain langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
)

print("Chunker initialized successfully.")

Chunker initialized successfully.


In [ ]:
all_chunks = []
metadata_list = []

for i, row in enumerate(small_cuad_sample):
  documents = row['documents']

  for doc in documents:
    chunks = text_splitter.split_text(doc)

    for chunk in chunks:
      all_chunks.append(chunk)
      metadata_list.append({"source_row": i, "question_id": row.get('id', 'unknown')})

print(f"Successfully split {len(small_cuad_sample)} examples into {len(all_chunks)} individual chunks!")
print("\n--- Sample Chunk ---")
print(all_chunks[0])

Successfully split 100 examples into 6746 individual chunks!

--- Sample Chunk ---
Exhibit 10.19   STRATEGIC ALLIANCE AGREEMENT

Among   SUMITOMO MITSUI BANKING CORPORATION, SMBC NIKKO SECURITIES INC.   And   MOELIS & COMPANY HOLDINGS LP, MOELIS & COMPANY HOLDINGS GP LLC   Dated December 27, 2011





  TABLE OF CONTENTS                                               ARTICLE I.       CERTAIN DEFINITIONS; INTERPRETATION.       1.1   Certain Definitions   2   1.2   Interpretations   5               ARTICLE II.       STRATEGIC ALLIANCE.       2.1   Strategic Alliance   6   2.2   Obligations of the Parties   6               ARTICLE III.       SCOPE.       3.1   Scope   6   3.2   Covered Businesses   6   3.3   Covered Regions   6   3.4   Japanese Companies   7   3.5   Client   7   3.6   Corporate Lending Business   7               ARTICLE IV.       FEE ALLOCATION.       4.1   General Allocation   7   4.2   Certain Moelis Holdings Sell-side Assignments   7


**Embedding and Vector Storage.**

In [ ]:
!pip install langchain-chroma langchain-huggingface chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling openteleme

In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Define an ABSOLUTE path in Colab's local storage
db_path = "/content/chroma_db"

# 2. Explicitly create the directory before Chroma tries to use it
os.makedirs(db_path, exist_ok=True)

print("Loading embedding model (this might take a minute)...")
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

print(f"Initializing Chroma DB at {db_path} and ingesting documents...")

# 3. Update the persist_directory
vectorstore = Chroma.from_texts(
    texts=all_chunks,
    embedding=embedding_model,
    metadatas=metadata_list,
    persist_directory=db_path,  # <--- Pointing to the local absolute path
    collection_name="cuad_baseline"
)

print(f"Successfully embedded and stored {len(all_chunks)} chunks in Chroma DB!")

Loading embedding model (this might take a minute)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Initializing Chroma DB at /content/chroma_db and ingesting documents...
Successfully embedded and stored 6746 chunks in Chroma DB!


In [ ]:
# A sample query relevant to legal contracts (CUAD domain)
test_query = "What is the governing law of this agreement?"

# Retrieve the top 3 most similar chunks
retrieved_docs = vectorstore.similarity_search(test_query, k=3)

print(f"Top results for: '{test_query}'\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Result {i+1} ---")
    print(f"Metadata: {doc.metadata}")
    print(f"Content: {doc.page_content}\n")

Top results for: 'What is the governing law of this agreement?'

--- Result 1 ---
Metadata: {'source_row': 4, 'question_id': 'RitterPharmaceuticalsInc_20200313_S-4A_EX-10.54_12055220_EX-10.54_Development Agreement__Agreement Date'}
Content: of the Parties. 15.7. Governing Law. This Agreement shall be governed by, and construed and interpreted in accordance with, the laws of the State of Delaware, without reference to its conflicts of laws principles. The parties agree that the United Nations Convention on Contracts for the International Sale of Goods shall be inapplicable to this Agreement. 15.8. Attorney Fees. If litigation becomes necessary to enforce the provisions of this Agreement, the successful Party shall be entitled to recover from the other Party reasonable expenses, including attorneys' and other professional fees, in addition to any other available remedies. 26

--- Result 2 ---
Metadata: {'source_row': 23, 'question_id': 'SONUSCORP_03_12_1997-EX-10.11-SPONSORSHIP AGREEMENT

**Generation**

In [ ]:
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 5.3 MB/s eta 0:00:00


**Load the LLM into the GPU**

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

print("Downloading LLM (this will take 1-2 minutes)...")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer and model directly onto the GPU
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16 # Loads in 16-bit to save memory
)

# Create a text-generation pipeline
text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,       # Limit answer length
    temperature=0.1,          # Keep answers factual and less creative
    do_sample=True,
    return_full_text=False    # Only return the answer, not the prompt
)

# Wrap it for LangChain
llm = HuggingFacePipeline(pipeline=text_pipeline)
print("LLM successfully loaded and ready!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM successfully loaded and ready!


**Build the RAG Chain**

In [ ]:
!pip install langchain langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetr

In [ ]:
import langchain
print(langchain.__file__)

import pkgutil
print(any(m.name == "chains" for m in pkgutil.iter_modules(langchain.__path__)))

/usr/local/lib/python3.12/dist-packages/langchain/__init__.py
False


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

# 1. Turn your Chroma DB into a "Retriever"
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 2. Draft the strict Prompt Template
# Note: LCEL traditionally uses {question} instead of {input}
prompt_template = """You are a legal assistant. Use the following pieces of retrieved context to answer the question.
If the answer is not contained in the context, say "I cannot answer this based on the provided documents."
Do not make up an answer. Keep your response concise.

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(prompt_template)

# Helper function to extract text from the Document objects
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Construct the pure LCEL Chain
# This fetches the docs, formats them, passes them to the prompt, and generates the answer
rag_chain = (
    RunnableParallel(
        {"context": retriever, "question": RunnablePassthrough()}
    )
    .assign(
        answer=(
            RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
            | prompt
            | llm
            | StrOutputParser()
        )
    )
)

print("Modern LCEL RAG Chain constructed successfully!")

Modern LCEL RAG Chain constructed successfully!


In [ ]:
query = "What is the governing law of the agreement?"

print(f"User Query: {query}\n")
print("Thinking...\n")

# Invoke the LCEL chain directly with a string
response = rag_chain.invoke(query)

print("--- GENERATED ANSWER ---")
print(response["answer"])

print("\n--- SOURCE DOCUMENTS USED ---")
for i, doc in enumerate(response["context"]):
    print(f"\nSource {i+1}:")
    print(f"Metadata: {doc.metadata}")
    print(f"Text snippet: {doc.page_content[:200]}...")

User Query: What is the governing law of the agreement?

Thinking...



[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- GENERATED ANSWER ---
 The governing law of the agreement is the laws of the Province of British Columbia, Canada. According to the second paragraph of the first set of context, it states: "This Agreement shall be governed by and construed in accordance with the laws of the Province of British Columbia (without regard to its conflict of laws provisions)." Additionally, the third paragraph of the same document specifies: "The parties hereby irrevocably submit and attorn to the non-exclusive jurisdiction of such Courts to finally adjudicate or determine any suit, action, or proceeding arising out of or in relation to this Agreement." These statements clearly indicate that the laws of the Province of British Columbia, Canada, govern the agreement.

--- SOURCE DOCUMENTS USED ---

Source 1:
Metadata: {'source_row': 4, 'question_id': 'RitterPharmaceuticalsInc_20200313_S-4A_EX-10.54_12055220_EX-10.54_Development Agreement__Agreement Date'}
Text snippet: of the Parties. 15.7. Governing Law.

In [ ]:
import pandas as pd

# Let's define 3 test questions relevant to the CUAD (Legal) domain
test_questions = [
    "What is the governing law of the agreement?",
    "Can this agreement be terminated early?",
    "What are the confidentiality obligations?"
]

results = []

print("Starting Baseline Evaluation Run...\n")

for question in test_questions:
    print(f"Asking: {question}")

    # Run the query through your LCEL chain
    response = rag_chain.invoke(question)

    # Extract the generated answer and the source documents
    answer = response["answer"]
    retrieved_contexts = [doc.page_content for doc in response["context"]]

    # Save the results to a dictionary
    results.append({
        "question": question,
        "generated_answer": answer,
        "retrieved_context": "\n\n".join(retrieved_contexts)
    })

# Convert to a Pandas DataFrame for easy viewing and saving
df_baseline = pd.DataFrame(results)

print("\nEvaluation Run Complete! Here are the results:")
display(df_baseline)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting Baseline Evaluation Run...

Asking: What is the governing law of the agreement?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Asking: Can this agreement be terminated early?


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Asking: What are the confidentiality obligations?

Evaluation Run Complete! Here are the results:


,question,generated_answer,retrieved_context
0,What is the governing law of the agreement?,The governing law of the agreement is the law...,of the Parties. 15.7. Governing Law. This Agre...
1,Can this agreement be terminated early?,"Yes, this agreement can be terminated early u...",7. Term and Termination. 7.1. Term. The initia...
2,What are the confidentiality obligations?,The confidentiality obligations include takin...,- 2 -\n\n\n\n\n\nARTICLE IV CONFIDENTIALITY\n...


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Define the Evaluation Prompt for Adherence
adherence_prompt_template = """You are an expert impartial judge evaluating a Retrieval-Augmented Generation (RAG) system.
Your task is to evaluate the 'Adherence' (Faithfulness) of a generated answer.

Adherence measures whether the generated answer is strictly grounded in the provided context.
If the answer contains claims, facts, or numbers not present in the context, it is a hallucination and should receive a low score.

Question: {question}

Retrieved Context:
{context}

Generated Answer: {answer}

Assign a score from 1 to 10, where:
- 10 means the answer is perfectly grounded in the context (no hallucinations).
- 1 means the answer is completely made up or uses outside knowledge.

Output ONLY the integer score. Do not provide an explanation.
Score:"""

adherence_prompt = PromptTemplate.from_template(adherence_prompt_template)

# 2. Build the Evaluation Chain using LCEL
eval_chain = adherence_prompt | llm | StrOutputParser()

print("Evaluating Baseline Results for Adherence...\n")

scores = []

# 3. Iterate through your saved baseline results and score them
for index, row in df_baseline.iterrows():
    # Pass the data from the dataframe into the evaluation chain
    score_output = eval_chain.invoke({
        "question": row["question"],
        "context": row["retrieved_context"],
        "answer": row["generated_answer"]
    })

    # Clean up the output to get just the number
    clean_score = score_output.strip()
    print(f"Q: {row['question']}")
    print(f"Adherence Score: {clean_score}/10\n")
    scores.append(clean_score)

# 4. Save the scores back to your DataFrame
df_baseline["adherence_score"] = scores

print("Evaluation complete! Updated DataFrame:")
display(df_baseline)